# Perpendicularidade

**Módulo 06 — Geometria Espacial** · Notebook 03 de 18

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/leandrofreiredealmeida/curso_matematica_python/blob/main/06_geometria_espacial/0603a_perpendicularidade.ipynb)

**Pré-requisitos**
- Módulo 06 — Geometria Espacial: [`0601a_introducao_geometria_de_posicao.ipynb`](0601a_introducao_geometria_de_posicao.ipynb) (postulados, determinação de plano, retas reversas) e [`0602a_paralelismo.ipynb`](0602a_paralelismo.ipynb) (especialmente o ângulo entre duas retas e as retas ortogonais)
- Módulo 04 — Geometria Plana: [`0405a_congruencia_de_triangulos.ipynb`](../04_geometria_plana/0405a_congruencia_de_triangulos.ipynb) (casos LLL e LAL), [`0407a_perpendicularidade.ipynb`](../04_geometria_plana/0407a_perpendicularidade.ipynb) (mediatriz de um segmento) e [`0416a_triangulos_retangulos.ipynb`](../04_geometria_plana/0416a_triangulos_retangulos.ipynb) (Teorema de Pitágoras)

**Objetivos de aprendizagem** — ao final deste notebook você será capaz de:
- definir reta perpendicular a um plano e verificar a perpendicularidade usando duas retas concorrentes;
- aplicar as propriedades das retas e planos perpendiculares a situações reais;
- usar o Teorema das três perpendiculares para encontrar a menor distância de um ponto a uma reta no espaço;
- reconhecer e justificar quando dois planos são perpendiculares.

**Tempo estimado:** 70 a 90 minutos

### Antes de começar

Rode a célula abaixo antes de qualquer outra. Ela carrega o `utils_3d.py`, o arquivo com as funções de desenho deste módulo.

In [1]:
# Célula de preparação: deixa o notebook pronto para rodar no seu computador ou no Colab.
import sys

if "google.colab" in sys.modules:
    # No Colab, o notebook chega sozinho; baixamos o arquivo de desenhos do repositório.
    import urllib.request
    url = "https://raw.githubusercontent.com/leandrofreiredealmeida/curso_matematica_python/main/06_geometria_espacial/utils_3d.py"
    urllib.request.urlretrieve(url, "utils_3d.py")

import utils_3d as u3d
from ipywidgets import interact
import ipywidgets as widgets

## O poste que parece reto... de um lado só

Uma equipe está instalando um poste. Para conferir se ele ficou "em pé", um técnico encosta um esquadro entre o poste e o chão e confirma: ângulo de 90°. Trabalho terminado?

Ainda não! Visto de frente, o poste pode parecer perfeito e, visto de lado, estar torto. Um único esquadro só garante o ângulo reto **numa direção** do chão.

🕹️ Use os controles abaixo para inclinar o poste para a frente e para o lado, e troque o ponto de vista. Observe o que dizem os dois esquadros: um colocado na direção da frente e outro na direção lateral.

In [2]:
import math

# As coordenadas (x, y, z) abaixo servem só para posicionar o desenho.
VISTAS = ["De frente", "De lado", "3D"]
CAMERAS = {
    "De frente": (0, -2.5, 0.3),
    "De lado": (2.5, 0, 0.3),
    "3D": (1.6, 1.4, 1.0),
}


def desenhar_poste(inclinacao_frente: int, inclinacao_lado: int, vista: str) -> None:
    """Desenha o poste, o chão α e os dois esquadros (frente e lateral)."""
    chao = ((0, 0, 0), (0, 0, 1))  # o chão α, como (ponto, normal)
    ponto_O = (0, 0, 0)  # a base do poste

    # A tangente (módulo 05) serve só para inclinar o desenho do poste. Dividir w
    # pelo seu comprimento |w| deixa o poste sempre com comprimento 4.
    w = (
        math.tan(math.radians(inclinacao_frente)),
        math.tan(math.radians(inclinacao_lado)),
        1,
    )
    tamanho_de_w = math.sqrt(w[0] ** 2 + w[1] ** 2 + w[2] ** 2)
    T = tuple(4 * coordenada / tamanho_de_w for coordenada in w)  # o topo do poste

    # As duas retas do chão, passando por O, onde encostamos os esquadros.
    frente = (1, 0, 0)
    lado = (0, 1, 0)
    angulo_1 = u3d.medir_angulo_entre_retas(ponto_O, T, ponto_O, frente)
    angulo_2 = u3d.medir_angulo_entre_retas(ponto_O, T, ponto_O, lado)

    fig = u3d.criar_figura(titulo="O poste e os dois esquadros")
    fig = u3d.desenhar_plano(fig, *chao, rotulo="chão α", tamanho=8)
    fig = u3d.desenhar_reta(
        fig, ponto_O, frente, rotulo="frente", cor=u3d.CORES["reta"], comprimento=6,
        espessura=5,
    )
    fig = u3d.desenhar_reta(
        fig, ponto_O, lado, rotulo="lado", cor=u3d.CORES["reta_secundaria"],
        comprimento=6, espessura=5,
    )
    fig = u3d.desenhar_segmento(fig, ponto_O, T, cor=u3d.CORES["apoio"], espessura=10)
    fig = u3d.desenhar_ponto(fig, ponto_O, rotulo="O")

    # Cada esquadro: o quadradinho, se o ângulo for reto, ou o arco com a medida.
    fig = u3d.desenhar_angulo_entre_retas(
        fig, ponto_O, T, ponto_O, frente, rotulo=f"{angulo_1:.0f}°",
        cor=u3d.CORES["reta"],
    )
    fig = u3d.desenhar_angulo_entre_retas(
        fig, ponto_O, T, ponto_O, lado, rotulo=f"{angulo_2:.0f}°",
        cor=u3d.CORES["reta_secundaria"],
    )
    fig = u3d.definir_camera(fig, olho=CAMERAS[vista])
    u3d.mostrar(fig)

    print(f"Esquadro na direção da frente: {angulo_1:.0f}°")
    print(f"Esquadro na direção lateral: {angulo_2:.0f}°")
    retos = [round(angulo, 6) == 90 for angulo in (angulo_1, angulo_2)]
    if all(retos):
        print("Os dois esquadros marcam 90°: o poste está perpendicular ao chão.")
    elif any(retos):
        print(
            "Um esquadro marca 90°, o outro não: o poste está torto! "
            "Um esquadro só não basta."
        )
    else:
        print("Nenhum esquadro marca 90°: o poste está torto.")


interact(
    desenhar_poste,
    inclinacao_frente=widgets.IntSlider(
        min=-20, max=20, step=5, value=0, description="inclinação para a frente (°)",
        continuous_update=False,
    ),
    inclinacao_lado=widgets.IntSlider(
        min=-20, max=20, step=5, value=10, description="inclinação lateral (°)",
        continuous_update=False,
    ),
    vista=widgets.RadioButtons(options=VISTAS, value="De frente", description="Vista"),
);

interactive(children=(IntSlider(value=0, continuous_update=False, description='inclinação para a frente (°)', …

Repare que, com a vista "De frente", você não enxerga a inclinação lateral. Um esquadro só pode enganar, mas **dois esquadros em direções que se cruzam** resolvem o problema. Neste notebook você vai descobrir por que duas direções bastam, e vai ver que essa ideia aparece de novo e de novo na Geometria Espacial.

> 📎 **Conexão.** Lembra do nível de bolha do notebook 02 ([`0602a_paralelismo.ipynb`](0602a_paralelismo.ipynb))? Lá, duas direções concorrentes bastavam para garantir que um piso está paralelo ao chão. Aqui é a mesma ideia, agora para perpendicularidade.

## 1. Reta perpendicular a um plano

> 📌 **Definição.** Uma reta $r$ é **perpendicular** a um plano $\alpha$ quando ela é concorrente com $\alpha$ (num ponto $O$, chamado **pé da perpendicular**) e forma ângulo reto com **todas** as retas de $\alpha$. Notação: $r \perp \alpha$.

Lembre-se do notebook 02:
- as retas de $\alpha$ que passam por $O$ são **perpendiculares** a $r$ (concorrentes a 90°);
- as retas de $\alpha$ que não passam por $O$ são **ortogonais** a $r$ (reversas a 90°).

Uma reta concorrente com o plano que **não** é perpendicular a ele é chamada **oblíqua** ao plano.

**No dia a dia:** um poste bem instalado, as colunas de um prédio, o fio de prumo de um pedreiro e o mastro de uma bandeira são perpendiculares ao chão. Uma escada encostada na parede é oblíqua ao chão.

> 🕵️ **Curiosidade histórica.** O **fio de prumo** (um peso pendurado num fio) é uma das ferramentas mais antigas da construção: já era usado por egípcios e romanos. Pela gravidade, o fio fica sempre na vertical, ou seja, perpendicular ao plano horizontal. Uma construção famosa em que a perpendicularidade "falhou" é a **Torre de Pisa**, na Itália: ela começou a inclinar ainda durante a construção, no século XII, porque o terreno de um dos lados era mole demais para o peso da torre.

🕹️ Gire a reta $c$ dentro do plano $\alpha$. Com $c$ passando pelo pé $O$, ela é perpendicular a $r$; desloque-a para fora de $O$ e ela vira ortogonal a $r$. Em todos os casos, o ângulo é de 90°.

In [3]:
def desenhar_retas_do_plano(angulo_de_c: int, deslocamento_de_c: float) -> None:
    """Desenha r ⊥ α no pé O e uma reta c de α, que gira e se desloca."""
    alfa = ((0, 0, 0), (0, 0, 1))  # o plano α, como (ponto, normal)
    ponto_O = (0, 0, 0)
    r1, r2 = ponto_O, (0, 0, 1)  # dois pontos de r

    # Seno e cosseno (módulo 05) servem só para girar a reta c dentro de α e para
    # deslocá-la "de lado", na direção perpendicular a ela mesma.
    theta = math.radians(angulo_de_c)
    c1 = (-deslocamento_de_c * math.sin(theta), deslocamento_de_c * math.cos(theta), 0)
    c2 = (c1[0] + math.cos(theta), c1[1] + math.sin(theta), 0)

    fig = u3d.criar_figura(titulo="r ⊥ α e uma reta c de α")
    fig = u3d.desenhar_plano(fig, *alfa, rotulo="α", tamanho=7)
    fig = u3d.desenhar_perpendicular_ao_plano(
        fig, alfa, r2, rotulo="r", cor=u3d.CORES["reta"], comprimento=6, espessura=6
    )
    fig = u3d.desenhar_ponto(fig, ponto_O, rotulo="O")
    fig = u3d.desenhar_reta_por_dois_pontos(
        fig, c1, c2, rotulo="c", cor=u3d.CORES["reta_secundaria"], comprimento=6,
        espessura=6,
    )
    # Se c for reversa a r, a função também desenha (tracejada) a paralela a c por O.
    fig = u3d.desenhar_angulo_entre_retas(fig, r1, r2, c1, c2)
    u3d.mostrar(fig)

    posicao = u3d.classificar_posicao_retas(r1, r2, c1, c2)
    angulo = u3d.medir_angulo_entre_retas(r1, r2, c1, c2)
    nome = "perpendiculares" if posicao == "concorrentes" else "ortogonais"
    print(f"Posição de r e c: {posicao}")
    print(f"Ângulo entre r e c: {angulo:.0f}° → r e c são {nome}")


interact(
    desenhar_retas_do_plano,
    angulo_de_c=widgets.IntSlider(
        min=0, max=180, step=15, value=30, description="ângulo de c (°)",
        continuous_update=False,
    ),
    deslocamento_de_c=widgets.FloatSlider(
        min=0, max=2, step=0.5, value=0, description="deslocamento de c",
        continuous_update=False,
    ),
);

interactive(children=(IntSlider(value=30, continuous_update=False, description='ângulo de c (°)', max=180, ste…

**Pare e pense**

1. No cubo, a aresta $AE$ é perpendicular ao plano da base $ABCD$. Ela é perpendicular ou ortogonal à aresta $BC$? E à aresta $AD$?
2. Uma reta perpendicular a um plano pode ser paralela a alguma reta desse plano?

<details>
<summary>Ver respostas</summary>

1. Com $AD$, ela é **perpendicular** (as duas passam por $A$, o pé da perpendicular). Com $BC$, ela é **ortogonal** (são reversas e formam 90°).
2. Não. Ela forma 90° com todas as retas do plano, e retas paralelas formam 0°.
</details>

## 2. Como verificar a perpendicularidade: o teorema fundamental

Pela definição, para garantir que uma reta é perpendicular a um plano teríamos que conferir **infinitas** retas do plano. Na prática, isso é impossível. O teorema abaixo mostra que bastam **duas**, exatamente como no gancho.

**Teorema fundamental da perpendicularidade**
> Se uma reta forma ângulo reto com **duas retas concorrentes** de um plano, então ela é perpendicular ao plano.

As duas retas podem ser perpendiculares ou ortogonais à reta dada; o que importa é que elas sejam concorrentes entre si.

> ⚠️ **Erro comum.** Conferir uma direção só (um esquadro só). Uma reta pode formar 90° com uma reta do plano e ser oblíqua ao plano. Também não adianta usar duas retas **paralelas**: elas precisam ser **concorrentes**.

🎯 **Aplicação.** É por isso que, na instalação de postes, mastros e pilares, o técnico confere o prumo em **duas direções diferentes**. E é por isso que um esquadro de carpinteiro, encostado em duas tábuas que se cruzam no chão, é suficiente para garantir que uma coluna ficou em pé.

<details>
<summary>Ver a demonstração (com os passos ilustrados no widget abaixo)</summary>

Seja $r$ perpendicular às retas $a$ e $b$ de $\alpha$, concorrentes em $O$ (se as retas dadas não passassem por $O$, bastaria trocá-las por paralelas que passam por $O$; o ângulo com $r$ não muda). Seja $c$ uma reta **qualquer** de $\alpha$ passando por $O$. Queremos mostrar que $r \perp c$.

1. Marque em $r$ dois pontos $P$ e $P'$, um de cada lado de $\alpha$, com $OP = OP'$.
2. Trace em $\alpha$ uma reta que corta $a$, $b$ e $c$ nos pontos $A$, $B$ e $C$.
3. No plano de $r$ e $a$, a reta $a$ é perpendicular a $PP'$ no seu ponto médio $O$ (é a mediatriz de $PP'$). Então $PA = P'A$. Do mesmo modo, $PB = P'B$.
4. Os triângulos $PAB$ e $P'AB$ têm os três lados iguais (caso LLL), logo os ângulos $P\hat{A}B$ e $P'\hat{A}B$ são iguais.
5. Então os triângulos $PAC$ e $P'AC$ são congruentes (caso LAL), e $PC = P'C$.
6. Assim, $C$ está à mesma distância de $P$ e de $P'$: ele está na mediatriz de $PP'$ no plano de $r$ e $c$. Essa mediatriz passa por $O$ e é perpendicular a $PP'$. Portanto $c \perp r$.

Como $c$ era qualquer, $r$ é perpendicular a todas as retas de $\alpha$ por $O$ e, por consequência, forma 90° com todas as retas de $\alpha$. ∎
</details>

🕹️ Avance os passos no widget para ver a demonstração acontecer. Acompanhe os comprimentos impressos: eles confirmam as igualdades de cada passo.

In [ ]:
# Os dados da demonstração (as coordenadas servem só para posicionar o desenho).
# Seno e cosseno (módulo 05) servem só para dar as direções das retas b e c em α.
ponto_O = (0, 0, 0)
ALFA = ((0, 0, 0), (0, 0, 1))
TOPO_DE_R = (0, 0, 1)  # um segundo ponto de r, para desenhar e medir ângulos
DIRECAO_A = (1, 0, 0)
DIRECAO_B = (math.cos(math.radians(70)), math.sin(math.radians(70)), 0)
DIRECAO_C = (math.cos(math.radians(35)), math.sin(math.radians(35)), 0)

ponto_P = (0, 0, 1.5)
ponto_P_ = (0, 0, -1.5)  # o ponto P'
ponto_A = (2, 0, 0)
ponto_B = tuple(2 * coordenada for coordenada in DIRECAO_B)
# C é o ponto onde a reta AB encontra o plano vertical que contém r e c.
plano_de_r_e_c = u3d.plano_por_tres_pontos(ponto_O, TOPO_DE_R, DIRECAO_C)
ponto_C = u3d.intersecao_reta_plano(ponto_A, ponto_B, plano_de_r_e_c)


def desenhar_demonstracao(passo: int) -> None:
    """Desenha a demonstração do teorema fundamental, até o passo escolhido."""
    medir = u3d.medir_distancia
    fig = u3d.criar_figura(titulo=f"Teorema fundamental: passo {passo}")
    fig = u3d.desenhar_plano(fig, *ALFA, rotulo="α", tamanho=6, opacidade=0.25)
    fig = u3d.desenhar_reta(
        fig, ponto_O, TOPO_DE_R, rotulo="r", cor=u3d.CORES["plano_terciario"],
        comprimento=5, espessura=6,
    )
    for direcao, nome in [(DIRECAO_A, "a"), (DIRECAO_B, "b")]:
        fig = u3d.desenhar_reta(
            fig, ponto_O, direcao, rotulo=nome, cor=u3d.CORES["apoio"], comprimento=5.5,
            espessura=4,
        )
        fig = u3d.desenhar_angulo_reto(fig, ponto_O, TOPO_DE_R, direcao)
    fig = u3d.desenhar_reta(
        fig, ponto_O, DIRECAO_C, rotulo="c", cor=u3d.CORES["plano_secundario"],
        comprimento=5.5, espessura=4, tracejado=True,
    )
    fig = u3d.desenhar_ponto(fig, ponto_O, rotulo="O")
    linhas = [
        (
            "Passo 1: r forma 90° com a e com b, que são concorrentes em O. "
            "Queremos mostrar que r também forma 90° com c."
        )
    ]

    if passo >= 2:
        fig = u3d.desenhar_pontos(fig, [ponto_P, ponto_P_], rotulos=["P", "P'"])
        linhas.append(
            f"Passo 2: OP = {medir(ponto_O, ponto_P):.3f} e "
            f"OP' = {medir(ponto_O, ponto_P_):.3f}: O é o ponto médio de PP'."
        )
    if passo >= 3:
        fig = u3d.desenhar_segmento(
            fig, ponto_A, ponto_B, cor=u3d.CORES["destaque"], espessura=5
        )
        fig = u3d.desenhar_pontos(
            fig, [ponto_A, ponto_B, ponto_C], rotulos=["A", "B", "C"]
        )
        linhas.append(
            "Passo 3: a reta AB, traçada em α, corta a, b e c nos pontos A, B e C."
        )
    if passo >= 4:
        for vertice in (ponto_A, ponto_B):
            fig = u3d.desenhar_segmento(fig, ponto_P, vertice, cor=u3d.CORES["reta"])
            fig = u3d.desenhar_segmento(
                fig, ponto_P_, vertice, cor=u3d.CORES["reta_secundaria"]
            )
        linhas.append(
            f"Passo 4: PA = {medir(ponto_P, ponto_A):.3f} e "
            f"P'A = {medir(ponto_P_, ponto_A):.3f}: iguais, pois a é mediatriz de PP'."
        )
        linhas.append(
            f"         PB = {medir(ponto_P, ponto_B):.3f} e "
            f"P'B = {medir(ponto_P_, ponto_B):.3f}: iguais, pois b é mediatriz de PP'."
        )
        linhas.append("         Os triângulos PAB e P'AB têm os três lados iguais.")
        linhas.append("         Pelo caso LLL, eles são congruentes.")
    if passo >= 5:
        fig = u3d.desenhar_segmento(fig, ponto_P, ponto_C, cor=u3d.CORES["reta"])
        fig = u3d.desenhar_segmento(
            fig, ponto_P_, ponto_C, cor=u3d.CORES["reta_secundaria"]
        )
        fig = u3d.desenhar_angulo_reto(fig, ponto_O, TOPO_DE_R, DIRECAO_C)
        linhas.append(
            f"Passo 5: PC = {medir(ponto_P, ponto_C):.3f} e "
            f"P'C = {medir(ponto_P_, ponto_C):.3f}: iguais (LAL em PAC e P'AC)."
        )
        linhas.append("         C está na mediatriz de PP', e por isso c ⊥ r.")
        angulo = u3d.medir_angulo_entre_retas(ponto_O, TOPO_DE_R, ponto_O, DIRECAO_C)
        linhas.append(f"Ângulo entre r e c: {angulo:.1f}°")
    u3d.mostrar(fig)
    print("\n".join(linhas))


interact(
    desenhar_demonstracao,
    passo=widgets.IntSlider(
        min=1, max=5, step=1, value=1, description="passo", continuous_update=False
    ),
);

interactive(children=(IntSlider(value=1, continuous_update=False, description='passo', max=5, min=1), Output()…

**Pare e pense**

1. No cubo, a diagonal $AC$ da base é perpendicular ao plano $BDHF$? Use o teorema: procure duas retas concorrentes de $BDHF$ que formam 90° com $AC$.
2. Por que o teorema exige que as duas retas sejam concorrentes? Dê um exemplo de reta oblíqua a um plano que forma 90° com duas retas paralelas desse plano.

<details>
<summary>Ver respostas</summary>

1. Sim. $AC \perp BD$ (diagonais do quadrado da base) e $AC$ é ortogonal a $BF$ (pois $BF$ é perpendicular à base, e então forma 90° com todas as retas da base). Como $BD$ e $BF$ são concorrentes em $B$, $AC \perp BDHF$.
2. Duas retas paralelas têm a mesma direção, então só "conferem" uma direção. Exemplo: no plano da base $ABCD$, as retas $AB$ e $DC$ são paralelas; a diagonal $AH$ da face $ADHE$ forma 90° com as duas, mas é oblíqua à base.
</details>

## 3. Propriedades das retas e planos perpendiculares

A partir da definição e do teorema fundamental, demonstram-se as propriedades abaixo.

1. **Plano perpendicular por um ponto.** Por um ponto dado passa um único plano perpendicular a uma reta dada.
2. **Reta perpendicular por um ponto.** Por um ponto dado passa uma única reta perpendicular a um plano dado.
3. **Duas retas perpendiculares a um mesmo plano são paralelas.**
4. **Dois planos perpendiculares a uma mesma reta são paralelos.**
5. Se $r \perp \alpha$ e $s \parallel r$, então $s \perp \alpha$.
6. Se $\alpha \parallel \beta$ e $r \perp \alpha$, então $r \perp \beta$.

🎯 **Aplicações**
- **Propriedade 3:** as colunas de um prédio, todas perpendiculares ao chão, são paralelas entre si. Os postes de uma rua plana também.
- **Propriedade 4:** os andares de um prédio são todos perpendiculares às colunas, e por isso são paralelos entre si (os níveis de voo do notebook 02 funcionam do mesmo jeito em relação à vertical).
- **Propriedade 6:** um elevador que sobe perpendicular ao térreo também é perpendicular a todos os andares.

> ⚠️ **Erro comum.** Achar que "duas retas perpendiculares a uma mesma **reta** são paralelas". No plano, isso é verdade; no espaço, **não**! Elas podem ser paralelas, concorrentes ou reversas. Compare com a propriedade 3, que fala de duas retas perpendiculares a um mesmo **plano**.

🕹️ Veja no cubo: todas as retas abaixo formam 90° com a aresta $AB$. Escolha um par e observe a posição relativa.

In [5]:
# Os vértices do cubo, com os nomes do módulo (A, B, C, D embaixo; E, F, G, H em cima).
vertices = u3d.vertices_do_cubo()
A, B, C, D, E, F, G, H = (vertices[nome] for nome in "ABCDEFGH")

PARES_PERPENDICULARES_A_AB = ["AE e BF", "AD e AE", "AE e BC", "AD e BF"]


def desenhar_perpendiculares_a_ab(par: str) -> None:
    """Desenha o cubo, a aresta AB e um par de retas que formam 90° com AB."""
    nome_1, nome_2 = par.split(" e ")
    p1, p2 = vertices[nome_1[0]], vertices[nome_1[1]]
    q1, q2 = vertices[nome_2[0]], vertices[nome_2[1]]

    fig = u3d.criar_figura(titulo=f"{nome_1} e {nome_2}: as duas formam 90° com AB")
    fig = u3d.desenhar_cubo(fig)
    fig = u3d.desenhar_reta_por_dois_pontos(
        fig, A, B, rotulo="AB", cor=u3d.CORES["destaque"], comprimento=6, espessura=9
    )
    fig = u3d.desenhar_reta_por_dois_pontos(
        fig, p1, p2, rotulo=nome_1, cor=u3d.CORES["reta"], comprimento=6, espessura=6
    )
    fig = u3d.desenhar_reta_por_dois_pontos(
        fig, q1, q2, rotulo=nome_2, cor=u3d.CORES["reta_secundaria"], comprimento=6,
        espessura=6,
    )
    fig = u3d.desenhar_angulo_entre_retas(fig, p1, p2, A, B, cor=u3d.CORES["reta"])
    fig = u3d.desenhar_angulo_entre_retas(
        fig, q1, q2, A, B, cor=u3d.CORES["reta_secundaria"]
    )
    u3d.mostrar(fig)

    for nome, (x1, x2) in [(nome_1, (p1, p2)), (nome_2, (q1, q2))]:
        angulo = u3d.medir_angulo_entre_retas(x1, x2, A, B)
        print(f"Ângulo entre {nome} e AB: {angulo:.0f}°")
    posicao = u3d.classificar_posicao_retas(p1, p2, q1, q2)
    print(f"Posição de {nome_1} e {nome_2}: {posicao}")
    print(
        "Duas retas perpendiculares (ou ortogonais) a uma mesma reta podem ser "
        "paralelas, concorrentes ou reversas."
    )


interact(
    desenhar_perpendiculares_a_ab,
    par=widgets.Dropdown(
        options=PARES_PERPENDICULARES_A_AB, value="AE e BF", description="Par"
    ),
);

interactive(children=(Dropdown(description='Par', options=('AE e BF', 'AD e AE', 'AE e BC', 'AD e BF'), value=…

Agora a propriedade 3 num prédio: quatro colunas perpendiculares ao chão sustentam uma laje paralela a ele.

In [6]:
chao = ((1.5, 1, 0), (0, 0, 1))
laje = ((1.5, 1, 3), (0, 0, 1))  # plano paralelo ao chão, 3 unidades acima
pes_das_colunas = [(0, 0, 0), (3, 0, 0), (3, 2, 0), (0, 2, 0)]
# Um segundo ponto de cada coluna, 1 unidade acima do pé (para classificar as retas).
topos = [(x, y, 1) for x, y, _ in pes_das_colunas]

fig = u3d.criar_figura(
    titulo="Colunas perpendiculares ao chão são paralelas entre si "
    "(e perpendiculares à laje)"
)
fig = u3d.desenhar_plano(fig, *chao, rotulo="chão", tamanho=6, opacidade=0.25)
fig = u3d.desenhar_plano(
    fig, *laje, rotulo="laje", cor=u3d.CORES["plano_secundario"], tamanho=6,
    opacidade=0.25,
)
# Como toda reta, cada coluna é desenhada como um trecho centrado no pé: a parte
# abaixo do chão lembra que a reta continua.
for numero, topo in enumerate(topos, start=1):
    fig = u3d.desenhar_perpendicular_ao_plano(
        fig, chao, topo, rotulo=f"coluna {numero}", cor=u3d.CORES["reta"],
        comprimento=7, espessura=7,
    )
u3d.mostrar(fig)

pe_1, pe_2, pe_3 = pes_das_colunas[:3]
topo_1, topo_2, topo_3 = topos[:3]
print(f"Colunas 1 e 2: {u3d.classificar_posicao_retas(pe_1, topo_1, pe_2, topo_2)}")
print(f"Colunas 1 e 3: {u3d.classificar_posicao_retas(pe_1, topo_1, pe_3, topo_3)}")
print(
    "A coluna 1 é perpendicular à laje? "
    f"{u3d.reta_perpendicular_ao_plano(pe_1, topo_1, laje)}"
)

Colunas 1 e 2: paralelas
Colunas 1 e 3: paralelas
A coluna 1 é perpendicular à laje? True


**Pare e pense**

1. Se uma reta $r$ é perpendicular a um plano $\alpha$, quantas retas perpendiculares a $\alpha$ passam por um ponto $P$ de $r$? Qual é a relação delas com $r$?
2. No cubo, a reta $AB$ é perpendicular ao plano $BCGF$. Use a propriedade 6 para descobrir a qual outra face do cubo $AB$ também é perpendicular.

<details>
<summary>Ver respostas</summary>

1. Uma única (propriedade 2), e é a própria reta $r$.
2. À face $ADHE$, que é paralela a $BCGF$.
</details>

## 4. Teorema das três perpendiculares

Um eletricista precisa ligar um cabo do **topo de um poste** até o **meio-fio** de uma rua que passa ao lado. Qual é o cabo mais curto possível, e em que ponto do meio-fio ele deve ser preso?

O meio-fio é uma reta $s$ do chão, e o poste está fora dela. O teorema abaixo responde à pergunta.

**Teorema das três perpendiculares**
> Seja $r$ uma reta perpendicular a um plano $\alpha$ no ponto $O$, e seja $s$ uma reta de $\alpha$ que não passa por $O$. Se $M$ é o ponto de $s$ tal que $OM \perp s$, então, para qualquer ponto $P$ de $r$, temos $PM \perp s$.

As "três perpendiculares" são:
1. $r \perp \alpha$ (o poste e o chão);
2. $OM \perp s$ (no chão, do pé do poste até o meio-fio);
3. $PM \perp s$ (o cabo, do topo do poste até o meio-fio).

<details>
<summary>Ver demonstração</summary>

A reta $s$ forma 90° com $OM$ (hipótese) e forma 90° com $r$ (pois $r \perp \alpha$ e $s \subset \alpha$). Como $r$ e $OM$ são concorrentes em $O$, pelo teorema fundamental $s$ é perpendicular ao plano determinado por $r$ e $OM$. A reta $PM$ está nesse plano e passa por $M$, logo $PM \perp s$. ∎
</details>

**A menor distância.** O segmento $PM$ é o menor segmento que liga $P$ à reta $s$ (é a **distância** de $P$ até $s$). Como o triângulo $POM$ é retângulo em $O$, o Teorema de Pitágoras dá:
$$PM^2 = PO^2 + OM^2$$

**Vale também a recíproca:** se $r \perp \alpha$ em $O$ e $PM \perp s$, então $OM \perp s$.

🕹️ Mexa na altura do poste, na distância do pé do poste até o meio-fio e no ponto onde o cabo é preso. Compare o comprimento do cabo com o valor dado por Pitágoras quando o cabo está preso em $M$.

In [ ]:
def desenhar_tres_perpendiculares(
    altura: float, distancia_ao_meio_fio: float, posicao_do_cabo: float
) -> None:
    """Desenha o poste, o meio-fio s, o segmento OM e o cabo PQ."""
    # O chão fica centrado entre o pé do poste e o meio-fio, para os dois caberem.
    chao = ((0, distancia_ao_meio_fio / 2, 0), (0, 0, 1))
    ponto_O = (0, 0, 0)  # o pé do poste
    P = (0, 0, altura)  # o topo do poste

    # O meio-fio s: reta do chão que passa por (0, OM, 0), na direção "x" do desenho.
    s1 = (0, distancia_ao_meio_fio, 0)
    s2 = (1, distancia_ao_meio_fio, 0)
    M = u3d.pe_da_perpendicular_a_reta(ponto_O, s1, s2)
    Q = (M[0] + posicao_do_cabo, M[1], M[2])  # onde o cabo é preso
    vizinho_de_M = (M[0] + 1, M[1], M[2])  # um ponto de s, para marcar os ângulos

    fig = u3d.criar_figura(titulo="Teorema das três perpendiculares", altura=550)
    fig = u3d.desenhar_plano(fig, *chao, rotulo="α", tamanho=12, opacidade=0.2)
    fig = u3d.desenhar_perpendicular_ao_plano(
        fig, chao, P, rotulo="r", cor=u3d.CORES["reta"], comprimento=2 * altura + 2,
        espessura=3,
    )
    fig = u3d.desenhar_segmento(fig, ponto_O, P, cor=u3d.CORES["apoio"], espessura=10)
    fig = u3d.desenhar_reta(
        fig, s1, (1, 0, 0), rotulo="s (meio-fio)", cor=u3d.CORES["reta_secundaria"],
        comprimento=12, espessura=5,
    )

    # Segunda perpendicular: OM ⊥ s, no chão.
    fig = u3d.desenhar_segmento(fig, ponto_O, M, cor=u3d.CORES["apoio"], tracejado=True)
    fig = u3d.desenhar_angulo_reto(fig, M, ponto_O, vizinho_de_M)
    if posicao_do_cabo != 0:
        # PM tracejado, para comparar com o cabo preso fora de M.
        fig = u3d.desenhar_segmento(fig, P, M, cor=u3d.CORES["apoio"], tracejado=True)
    fig = u3d.desenhar_segmento(fig, P, Q, cor=u3d.CORES["destaque"], espessura=8)
    if posicao_do_cabo == 0:
        # Terceira perpendicular: PM ⊥ s.
        fig = u3d.desenhar_angulo_reto(fig, M, P, vizinho_de_M)
    fig = u3d.desenhar_pontos(fig, [ponto_O, P, M], rotulos=["O", "P", "M"])
    if posicao_do_cabo != 0:
        fig = u3d.desenhar_ponto(fig, Q, rotulo="Q", cor=u3d.CORES["destaque"])
    u3d.mostrar(fig)

    # Pitágoras no triângulo POM (retângulo em O)
    pm_pitagoras = (altura**2 + distancia_ao_meio_fio**2) ** 0.5

    cabo = u3d.medir_distancia(P, Q)
    angulo = u3d.medir_angulo_entre_retas(P, Q, s1, s2)
    print(f"Comprimento do cabo PQ (medido): {cabo:.3f} m")
    print(
        f"PM por Pitágoras: √({altura:g}² + {distancia_ao_meio_fio:g}²) = "
        f"{pm_pitagoras:.3f} m"
    )
    print(f"Ângulo entre o cabo e o meio-fio: {angulo:.1f}°")
    if posicao_do_cabo == 0:
        print("O cabo está preso em M: é o cabo mais curto e forma 90° com o meio-fio.")
    else:
        print("O cabo está fora de M: ele é mais longo que PM.")


interact(
    desenhar_tres_perpendiculares,
    altura=widgets.FloatSlider(
        min=1, max=6, step=0.5, value=4, description="PO (altura)",
        continuous_update=False,
    ),
    distancia_ao_meio_fio=widgets.FloatSlider(
        min=1, max=6, step=0.5, value=3, description="OM (até o meio-fio)",
        continuous_update=False,
    ),
    posicao_do_cabo=widgets.FloatSlider(
        min=-4, max=4, step=0.5, value=0, description="posição do cabo",
        continuous_update=False,
    ),
);

interactive(children=(FloatSlider(value=4.0, continuous_update=False, description='PO (altura)', max=6.0, min=…

### O cabo mais curto, em números

A tabela abaixo calcula o comprimento do cabo para vários pontos de fixação ao longo do meio-fio, com o poste de 4 m e o pé do poste a 3 m do meio-fio. Repare onde está o menor valor.

In [8]:
altura_do_poste = 4  # PO, em metros
distancia_ao_meio_fio = 3  # OM, em metros

# Pitágoras no triângulo POM (retângulo em O): PM² = PO² + OM².
pm = (altura_do_poste**2 + distancia_ao_meio_fio**2) ** 0.5

print(f"{'distância de Q até M (m)':^26}{'comprimento do cabo (m)':^26}")
for x in range(-4, 5):
    # Pelo teorema, PM ⊥ s, então o triângulo PMQ é retângulo em M:
    # PQ² = PM² + MQ² = PO² + OM² + x² (Pitágoras pela segunda vez).
    cabo = (pm**2 + x**2) ** 0.5
    marca = "  ← menor" if x == 0 else ""
    print(f"{abs(x):^26}{cabo:^26.3f}{marca}")

 distância de Q até M (m)  comprimento do cabo (m)  
            4                       6.403           
            3                       5.831           
            2                       5.385           
            1                       5.099           
            0                       5.000             ← menor
            1                       5.099           
            2                       5.385           
            3                       5.831           
            4                       6.403           


**Pare e pense**

1. Na tabela, qual é o menor cabo? Ele confere com o Pitágoras $\sqrt{4^2 + 3^2}$?
2. No cubo de aresta 2, a aresta $GC$ é perpendicular à base e $CB \perp AB$. Qual é o pé da perpendicular de $G$ até a reta $AB$, e qual é a distância de $G$ até $AB$?

<details>
<summary>Ver respostas</summary>

1. Sim: 5 m, quando o cabo é preso em $M$.
2. Pelo teorema das três perpendiculares (com $r = GC$, $O = C$, $s = AB$ e $M = B$), $GB \perp AB$. O pé é $B$ e a distância é $GB = \sqrt{2^2 + 2^2} = 2\sqrt{2} \approx 2{,}83$.
</details>

## 5. Planos perpendiculares

> 📌 **Definição.** Dois planos $\alpha$ e $\beta$ são **perpendiculares** quando um deles contém uma reta perpendicular ao outro. Notação: $\alpha \perp \beta$.

**No dia a dia:** as paredes de uma sala e o chão; as páginas de um livro em pé sobre a mesa, abertas em qualquer ângulo; as faces vizinhas de uma caixa de sapatos.

**Teorema**
> Se $\alpha \perp \beta$, então toda reta de $\alpha$ perpendicular à reta de interseção $i = \alpha \cap \beta$ é perpendicular a $\beta$.

Isso dá um jeito prático de construir: para levantar uma reta perpendicular ao chão encostada numa parede perpendicular a ele, basta traçar na parede uma reta perpendicular ao rodapé.

**Propriedades**
1. Se uma reta $r$ é perpendicular a $\beta$, **todo plano que contém $r$** é perpendicular a $\beta$.
2. Se dois planos secantes são ambos perpendiculares a um terceiro plano, então a reta de interseção deles também é perpendicular a esse terceiro plano.
3. Por uma reta **não perpendicular** a um plano passa um **único** plano perpendicular a ele.

🎯 **Aplicações**
- **Propriedade 1: a porta.** As dobradiças de uma porta ficam numa reta vertical, perpendicular ao chão. Por isso, em **qualquer** posição de abertura, a porta é perpendicular ao chão.
- **Propriedade 2: o canto da sala.** Duas paredes vizinhas são perpendiculares ao chão; por isso a quina entre elas é vertical.

> ⚠️ **Erro comum.**
> - Achar que, se $\alpha \perp \beta$, **toda** reta de $\alpha$ é perpendicular a $\beta$. Só as retas de $\alpha$ perpendiculares à interseção são. Uma escada desenhada na parede, inclinada, não é perpendicular ao chão.
> - Achar que, se $\alpha \perp \beta$ e $\beta \perp \gamma$, então $\alpha \parallel \gamma$. Duas paredes vizinhas são perpendiculares ao chão e se cortam!

🕹️ Abra e feche a porta abaixo.

In [ ]:
def desenhar_porta(abertura: int) -> None:
    """Desenha o chão, a parede, as dobradiças e a porta aberta no ângulo dado."""
    chao = ((0, 0, 0), (0, 0, 1))
    parede = u3d.plano_por_tres_pontos((0, 0, 0), (1, 0, 0), (0, 0, 1))

    # Seno e cosseno (módulo 05) servem só para girar o desenho da porta em torno
    # das dobradiças.
    theta = math.radians(abertura)
    canto_da_porta = (math.cos(theta), math.sin(theta), 0)
    porta = u3d.plano_por_tres_pontos((0, 0, 0), (0, 0, 1), canto_da_porta)

    fig = u3d.criar_figura(titulo=f"Uma porta aberta a {abertura}°")
    fig = u3d.desenhar_plano(fig, *chao, rotulo="chão", tamanho=10, opacidade=0.25)
    fig = u3d.desenhar_plano(
        fig, (0, 0, 2), parede[1], rotulo="parede", cor=u3d.CORES["plano_terciario"],
        opacidade=0.15, tamanho=8,
    )
    # A porta é um quadrado de lado 4 do plano da porta, com um lado nas dobradiças.
    centro_da_porta = (2 * math.cos(theta), 2 * math.sin(theta), 2)
    fig = u3d.desenhar_plano(
        fig, centro_da_porta, porta[1], rotulo="porta",
        cor=u3d.CORES["plano_secundario"], tamanho=4,
    )
    fig = u3d.desenhar_perpendicular_ao_plano(
        fig, chao, (0, 0, 1), rotulo="dobradiças", cor=u3d.CORES["destaque"],
        comprimento=8, espessura=7,
    )
    u3d.mostrar(fig)

    print(
        "A porta é perpendicular ao chão? "
        f"{u3d.planos_sao_perpendiculares(porta, chao)}"
    )
    print(
        "A porta é perpendicular à parede? "
        f"{u3d.planos_sao_perpendiculares(porta, parede)}"
    )


interact(
    desenhar_porta,
    abertura=widgets.IntSlider(
        min=0, max=180, step=15, value=60, description="abertura (°)",
        continuous_update=False,
    ),
);

interactive(children=(IntSlider(value=60, continuous_update=False, description='abertura (°)', max=180, step=1…

### Um único plano perpendicular por uma reta

A propriedade 3 diz que, por uma reta **não** perpendicular a um plano, passa um único plano perpendicular a ele. Pense numa vara inclinada fincada no chão: só existe um plano "em pé" que contém a vara, a parede imaginária que a vara forma com a vertical.

🕹️ Incline a reta $r$ abaixo e veja o plano perpendicular ao chão que a contém. O que acontece quando $r$ fica vertical?

In [ ]:
def desenhar_plano_em_pe(inclinacao: int, giro: int) -> None:
    """Desenha a reta r, inclinada, e o plano perpendicular ao chão que a contém."""
    chao = ((0, 0, 0), (0, 0, 1))
    ponto_O = (0, 0, 0)

    # Seno e cosseno (módulo 05) servem só para inclinar e girar o desenho de r:
    # "inclinacao" é o ângulo de r com o chão e "giro" diz para que lado r aponta.
    i, g = math.radians(inclinacao), math.radians(giro)
    r2 = (math.cos(i) * math.cos(g), math.cos(i) * math.sin(g), math.sin(i))

    fig = u3d.criar_figura(titulo="O plano perpendicular ao chão que contém r")
    fig = u3d.desenhar_plano(fig, *chao, rotulo="chão", tamanho=6, opacidade=0.25)
    try:
        fig = u3d.desenhar_plano_perpendicular_por_reta(
            fig, ponto_O, r2, chao, rotulo="β", cor=u3d.CORES["plano_secundario"],
            tamanho=4,
        )
        beta = u3d.plano_perpendicular_por_reta(ponto_O, r2, chao)
        mensagens = [
            "Existe um único plano que contém r e é perpendicular ao chão.",
            f"β é perpendicular ao chão? {u3d.planos_sao_perpendiculares(beta, chao)}",
        ]
    except ValueError as erro:
        # r é vertical: três dos infinitos planos verticais que contêm r. O ângulo
        # (módulo 05) serve só para girar cada plano em torno de r.
        cores = [
            u3d.CORES["plano"], u3d.CORES["plano_secundario"],
            u3d.CORES["plano_terciario"],
        ]
        for angulo_do_plano, cor in zip((0, 60, 120), cores, strict=True):
            phi = math.radians(angulo_do_plano)
            no_chao = (math.cos(phi), math.sin(phi), 0)
            plano = u3d.plano_por_tres_pontos(ponto_O, r2, no_chao)
            fig = u3d.desenhar_plano(
                fig, (0, 0, 1), plano[1], cor=cor, opacidade=0.15, tamanho=4
            )
        mensagens = [
            str(erro),
            (
                "Quando r é perpendicular ao chão, todos os planos que a contêm são "
                "perpendiculares ao chão (propriedade 1)."
            ),
        ]
    fig = u3d.desenhar_reta(
        fig, ponto_O, r2, rotulo="r", cor=u3d.CORES["reta"], comprimento=6, espessura=6
    )
    fig = u3d.desenhar_ponto(fig, ponto_O, rotulo="O")
    u3d.mostrar(fig)
    print("\n".join(mensagens))


interact(
    desenhar_plano_em_pe,
    inclinacao=widgets.IntSlider(
        min=30, max=90, step=15, value=45, description="inclinação (°)",
        continuous_update=False,
    ),
    giro=widgets.IntSlider(
        min=0, max=180, step=30, value=30, description="giro (°)",
        continuous_update=False,
    ),
);

interactive(children=(IntSlider(value=45, continuous_update=False, description='inclinação (°)', max=90, min=3…

**Pare e pense**

1. No cubo, os planos diagonais $ACGE$ e $BDHF$ são perpendiculares? Justifique com a definição.
2. Uma parede é perpendicular ao chão. Uma reta desenhada na parede, perpendicular ao rodapé, é perpendicular ao chão? E uma reta paralela ao rodapé?

<details>
<summary>Ver respostas</summary>

1. Sim. Na seção 2 vimos que $AC \perp BDHF$. Como $AC$ está contida em $ACGE$, pela definição os planos são perpendiculares.
2. A reta perpendicular ao rodapé é perpendicular ao chão (pelo teorema desta seção). A reta paralela ao rodapé é paralela ao chão, e não perpendicular.
</details>

## Resumo

**Reta perpendicular a plano:** $r \perp \alpha$ quando $r$ fura $\alpha$ e forma 90° com **todas** as retas de $\alpha$ (perpendicular às que passam pelo pé, ortogonal às outras). Se fura sem ser perpendicular, é **oblíqua**.

**Teorema fundamental:** $r$ forma 90° com **duas retas concorrentes** de $\alpha$ ⟹ $r \perp \alpha$.

**Propriedades**
- Por um ponto: um único plano perpendicular a uma reta; uma única reta perpendicular a um plano.
- $r \perp \alpha$ e $s \perp \alpha$ ⟹ $r \parallel s$.
- $\alpha \perp r$ e $\beta \perp r$ ⟹ $\alpha \parallel \beta$.
- $r \perp \alpha$ e $s \parallel r$ ⟹ $s \perp \alpha$.
- $\alpha \parallel \beta$ e $r \perp \alpha$ ⟹ $r \perp \beta$.
- ⚠️ Duas retas perpendiculares a uma mesma **reta** podem ser paralelas, concorrentes ou reversas.

**Três perpendiculares:** $r \perp \alpha$ em $O$, $s \subset \alpha$, $OM \perp s$ ⟹ $PM \perp s$ para todo $P \in r$.
Distância de $P$ até $s$: $PM = \sqrt{PO^2 + OM^2}$.

**Planos perpendiculares:** $\alpha \perp \beta$ quando um contém uma reta perpendicular ao outro.
- Retas de $\alpha$ perpendiculares à interseção são perpendiculares a $\beta$.
- Todo plano que contém uma reta perpendicular a $\beta$ é perpendicular a $\beta$ (a porta).
- Dois planos secantes perpendiculares a um terceiro ⟹ a interseção é perpendicular a ele (o canto da sala).
- Por uma reta não perpendicular a $\beta$ passa um único plano perpendicular a $\beta$.

## Exercícios

Responda preenchendo as variáveis indicadas em cada célula (troque cada `...` pela sua resposta) e rode a célula de verificação logo abaixo. Os exercícios estão em três níveis: **básico**, **intermediário** e **desafio**. Use o cubo $ABCDEFGH$ sempre que ajudar.

O gabarito comentado está em [`0603b_perpendicularidade_gabarito.ipynb`](0603b_perpendicularidade_gabarito.ipynb). Tente sozinho antes de olhar!

Use o cubo abaixo para consulta durante os exercícios. Gire a figura à vontade!

In [11]:
fig = u3d.criar_figura(titulo="Cubo ABCDEFGH")
fig = u3d.desenhar_cubo(fig)
u3d.mostrar(fig)

### Básico

In [ ]:
# Exercício Básico 1: verdadeiro ou falso? Troque cada ... por True ou False.
# (a) Se uma reta forma 90° com duas retas concorrentes de um plano, ela é
#     perpendicular ao plano.
# (b) Se uma reta é perpendicular a uma reta de um plano, ela é perpendicular ao plano.
# (c) Duas retas perpendiculares a um mesmo plano são paralelas.
# (d) Duas retas perpendiculares a uma mesma reta são paralelas.
# (e) Se dois planos são perpendiculares, toda reta de um deles é perpendicular ao
#     outro.
respostas_ex1 = {
    "a": ...,
    "b": ...,
    "c": ...,
    "d": ...,
    "e": ...,
}

print(respostas_ex1)

In [ ]:
# Verificação
gabarito_ex1 = {"a": True, "b": False, "c": True, "d": False, "e": False}
for item, certa in gabarito_ex1.items():
    assert respostas_ex1[item] == certa, (
        f"Tente de novo, revise: o item ({item}) não está certo; releia o teorema "
        "fundamental e os erros comuns (seção 2 - Como verificar a "
        "perpendicularidade: o teorema fundamental; seção 3 - Propriedades das retas "
        "e planos perpendiculares; seção 5 - Planos perpendiculares)."
    )
print("Certo!")

In [ ]:
# Exercício Básico 2: a reta é perpendicular ao plano? Troque cada ... por True ou
# False (use o cubo ABCDEFGH).
perpendiculares_ex2 = {
    "AE e ABCD": ...,
    "AB e BCGF": ...,
    "AG e ABCD": ...,
    "BF e EFGH": ...,
    "AC e BDHF": ...,
}

print(perpendiculares_ex2)

In [ ]:
# Verificação
gabarito_ex2 = {
    "AE e ABCD": True,
    "AB e BCGF": True,
    "AG e ABCD": False,
    "BF e EFGH": True,
    "AC e BDHF": True,
}
for par, certa in gabarito_ex2.items():
    assert perpendiculares_ex2[par] == certa, (
        f"Tente de novo, revise: para {par}, procure duas retas concorrentes do plano "
        "que formem 90° com a reta (seção 2 - Como verificar a perpendicularidade: o "
        "teorema fundamental)."
    )
print("Certo!")

Para o Exercício 3, veja o plano $BDE$ desenhado dentro do cubo.

In [12]:
fig = u3d.criar_figura(titulo="O plano BDE")
fig = u3d.desenhar_cubo(fig)
fig = u3d.desenhar_plano_por_tres_pontos(
    fig, B, D, E, rotulo="BDE", opacidade=0.3, tamanho=1
)
# O triângulo BDE, cujos lados são diagonais das faces.
for inicio, fim in [(B, D), (D, E), (E, B)]:
    fig = u3d.desenhar_segmento(fig, inicio, fim, cor=u3d.CORES["reta"], espessura=5)
u3d.mostrar(fig)

In [ ]:
# Exercício Básico 3: os planos são perpendiculares? Troque cada ... por True ou
# False (use o cubo ABCDEFGH).
perpendiculares_ex3 = {
    "ABCD e ABFE": ...,
    "ABCD e EFGH": ...,
    "ACGE e BDHF": ...,
    "ABFE e BCGF": ...,
    "BDE e ABCD": ...,
}

print(perpendiculares_ex3)

In [ ]:
# Verificação
gabarito_ex3 = {
    "ABCD e ABFE": True,
    "ABCD e EFGH": False,
    "ACGE e BDHF": True,
    "ABFE e BCGF": True,
    "BDE e ABCD": False,
}
for par, certa in gabarito_ex3.items():
    assert perpendiculares_ex3[par] == certa, (
        f"Tente de novo, revise: para {par}, procure, em um dos planos, uma reta "
        "perpendicular ao outro (seção 5 - Planos perpendiculares)."
    )
print("Certo!")

### Intermediário

In [ ]:
# Exercício Intermediário 4: um poste de 6 m está a 8 m do meio-fio (a distância do
# pé do poste até o meio-fio é medida perpendicularmente ao meio-fio). Qual é o
# comprimento do menor cabo que liga o topo do poste ao meio-fio? Escreva o cálculo
# em Python (não só o resultado).
altura_do_poste = 6
distancia_ao_meio_fio = 8

cabo_ex4 = ...  # em metros

print(cabo_ex4)

In [ ]:
# Verificação
assert cabo_ex4 is not ... and abs(cabo_ex4 - 10) <= 0.01, (
    "Tente de novo, revise: use o teorema das três perpendiculares e Pitágoras no "
    "triângulo POM (seção 4 - Teorema das três perpendiculares)."
)
print("Certo!")

In [ ]:
# Exercício Intermediário 5: num cubo de aresta 3, calcule a distância do vértice G
# à reta AB. Escreva o cálculo em Python (não só o resultado).
aresta = 3

distancia_ex5 = ...  # a distância de G até a reta AB

print(distancia_ex5)

In [ ]:
# Verificação
assert distancia_ex5 is not ... and abs(distancia_ex5 - 3 * 2**0.5) <= 0.01, (
    "Tente de novo, revise: GC é perpendicular à base e CB é perpendicular a AB: "
    "pelas três perpendiculares, GB é perpendicular a AB (seção 4 - Teorema das três "
    "perpendiculares)."
)
print("Certo!")

In [ ]:
# Exercício Intermediário 6: as retas abaixo formam 90° com a aresta AB do cubo.
# Classifique cada par com "paralelas", "concorrentes" ou "reversas".
posicoes_ex6 = {
    "AE e BF": ...,
    "AD e AE": ...,
    "AE e BC": ...,
}

print(posicoes_ex6)

In [ ]:
# Verificação
gabarito_ex6 = {
    "AE e BF": "paralelas",
    "AD e AE": "concorrentes",
    "AE e BC": "reversas",
}
for par, certa in gabarito_ex6.items():
    assert posicoes_ex6[par] == certa, (
        f"Tente de novo, revise: a posição de {par} não está certa; revise o widget "
        "das retas perpendiculares a AB (seção 3 - Propriedades das retas e planos "
        "perpendiculares)."
    )
print("Certo!")

### Desafio

In [ ]:
# Exercício Desafio 7: a diagonal e o plano BDE. Troque cada ... por True ou False.
# (a) A diagonal AG do cubo é perpendicular ao plano BDE?
# (b) Os planos ACGE e BDE são perpendiculares?
# Justifique para você mesmo com o teorema fundamental (seção 2) e com a definição de
# planos perpendiculares (seção 5).
ag_perpendicular_a_bde_ex7 = ...  # (a)
acge_perpendicular_a_bde_ex7 = ...  # (b)

print(ag_perpendicular_a_bde_ex7, acge_perpendicular_a_bde_ex7)

In [ ]:
# Verificação
assert ag_perpendicular_a_bde_ex7 is True, (
    "Tente de novo, revise: BD é perpendicular ao plano ACGE (por quê?), então BD "
    "forma 90° com AG. Faça o mesmo com BE e o plano ADGF (seção 2 - Como verificar a "
    "perpendicularidade: o teorema fundamental)."
)
assert acge_perpendicular_a_bde_ex7 is True, (
    "Tente de novo, revise: se AG ⊥ BDE e AG está contida em ACGE, use a definição de "
    "planos perpendiculares (seção 5 - Planos perpendiculares)."
)
print("Certo!")

# Conferência com as funções do utils_3d (que usam vetores, assunto do módulo 07).
v = u3d.vertices_do_cubo()
plano_bde = u3d.plano_por_tres_pontos(v["B"], v["D"], v["E"])
plano_acge = u3d.plano_por_tres_pontos(v["A"], v["C"], v["G"])
print("AG ⊥ BDE?", u3d.reta_perpendicular_ao_plano(v["A"], v["G"], plano_bde))
print("ACGE ⊥ BDE?", u3d.planos_sao_perpendiculares(plano_acge, plano_bde))

# O cubo com o plano BDE, a diagonal AG e o ponto onde ela fura o plano.
fig = u3d.criar_figura(titulo="A diagonal AG é perpendicular ao plano BDE")
fig = u3d.desenhar_cubo(fig)
fig = u3d.desenhar_plano_por_tres_pontos(
    fig, v["B"], v["D"], v["E"], rotulo="BDE", opacidade=0.3, tamanho=1
)
fig = u3d.desenhar_segmento(fig, v["A"], v["G"], cor=u3d.CORES["reta"], espessura=6)
fig = u3d.desenhar_intersecao_reta_plano(fig, v["A"], v["G"], plano_bde, rotulo="P")
u3d.mostrar(fig)

In [ ]:
# Exercício Desafio 8: em que fração da diagonal AG fica o ponto P onde ela fura o
# plano BDE? Complete o esqueleto abaixo usando as funções do utils_3d e responda
# com a fração AP / AG como número decimal.
vertices = u3d.vertices_do_cubo()
A, G = vertices["A"], vertices["G"]
B, D, E = vertices["B"], vertices["D"], vertices["E"]

plano_bde = ...  # complete aqui: use u3d.plano_por_tres_pontos com B, D e E
ponto_P = ...  # complete aqui: use u3d.intersecao_reta_plano com A, G e plano_bde
ap = ...  # complete aqui: use u3d.medir_distancia entre A e ponto_P
ag = ...  # complete aqui: use u3d.medir_distancia entre A e G

fracao_ex8 = ...  # complete aqui: AP / AG

print(ponto_P, fracao_ex8)

In [ ]:
# Verificação
assert fracao_ex8 is not ... and abs(fracao_ex8 - 1 / 3) <= 0.001, (
    "Tente de novo, revise: calcule AP e AG com medir_distancia e divida (seção 2 - "
    "Como verificar a perpendicularidade: o teorema fundamental)."
)
print("Certo!")
print(
    "Os planos BDE e CFH (que você viu no notebook 02) dividem a diagonal AG em três "
    "partes iguais! No notebook 04, você vai ver que AP é a distância do vértice A ao "
    "plano BDE."
)

## Conexões

**Isso depende de:**
- [`0601a_introducao_geometria_de_posicao.ipynb`](0601a_introducao_geometria_de_posicao.ipynb) (postulados, determinação de plano, posições relativas)
- [`0602a_paralelismo.ipynb`](0602a_paralelismo.ipynb) (ângulo entre retas, retas ortogonais, condição de paralelismo entre planos)
- [`0405a_congruencia_de_triangulos.ipynb`](../04_geometria_plana/0405a_congruencia_de_triangulos.ipynb) (casos LLL e LAL, usados na demonstração do teorema fundamental)
- [`0407a_perpendicularidade.ipynb`](../04_geometria_plana/0407a_perpendicularidade.ipynb) (mediatriz de um segmento)
- [`0416a_triangulos_retangulos.ipynb`](../04_geometria_plana/0416a_triangulos_retangulos.ipynb) (Teorema de Pitágoras, usado nas três perpendiculares)

**Isso será usado em:**
- [`0604a_projecoes_distancias_e_angulos.ipynb`](0604a_projecoes_distancias_e_angulos.ipynb) (projeção ortogonal, distância de ponto a plano, ângulo entre reta e plano)
- [`0605a_diedros.ipynb`](0605a_diedros.ipynb) (a seção reta do diedro e o diedro reto)
- 📌 Prismas e paralelepípedos (prismas retos e a diagonal do paralelepípedo)
- 📌 Pirâmide (altura da pirâmide e apótema: as três perpendiculares aparecem de novo)
- 📌 Geometria Analítica Plana e Espacial (vetor normal a um plano e produto escalar)